# 第4日 ノートブック02：YOLO モデルを学習する

# Day 4 Notebook 02: Training a YOLO Model

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. 学習の流れ（epoch・batch・loss）を説明できる | 1. Explain the training loop (epoch, batch, loss) |
| 2. ファインチューニングの意味を説明できる | 2. Explain what fine-tuning is |
| 3. CPU で現実的な時間で学習を実行できる | 3. Run a CPU-feasible training job |
| 4. 学習の進み具合を読み取れる | 4. Read training progress |
| 5. 学習結果（重み・設定・指標・ログ）を保存できる | 5. Save weights, config, metrics and logs |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 150 分 / 150 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第4日ノートブック01 を終え、`assets/yolo_dataset` が生成されていること。 / You finished Day 4 Notebook 01 and `assets/yolo_dataset` exists. |
| 必要なライブラリ / Required libraries | torch, ultralytics, opencv-python, numpy, matplotlib, pandas |
| 使用カーネル / Jupyter kernel | aivision |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 学習済み重みファイル | The trained weight file |
| 学習ログと指標 | Training logs and metrics |
| 学習設定の記録 | A record of the training configuration |

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師：学習の仕組みの説明 | Instructor: how training works |
| 09:00–10:30 | Part 1〜Part 3（学習の概念・準備） | Part 1-Part 3 (concepts, preparation) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4（学習の実行と観察） | Part 4 (run and observe training) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：結果の確認と午後の説明 | Instructor: results review and briefing |
| 13:50–16:50 | Part 5・演習・提出物 | Part 5, exercises and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("cv2", "opencv-python"),
    ("torch", "torch"),
    ("ultralytics", "ultralytics"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

In [ ]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

def show(images, titles=None, figsize=(14, 5)):
    """画像を並べて表示する / display images side by side."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    if titles is None:
        titles = [""] * len(images)
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, image, title in zip(axes, images, titles):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


def detections_to_frame(result):
    """検出結果を pandas の表にする / convert detections into a DataFrame.

    ``result.boxes.data`` の 6 列は
    (x1, y1, x2, y2, confidence, class_id) です。
    """
    columns = ["class_id", "name", "confidence", "x1", "y1", "x2", "y2"]
    boxes = result.boxes
    if len(boxes) == 0:
        return pd.DataFrame(columns=columns)

    data = boxes.data.cpu().numpy()
    return pd.DataFrame({
        "class_id": data[:, 5].astype(int),
        "name": [result.names[int(class_id)] for class_id in data[:, 5]],
        "confidence": data[:, 4].round(4),
        "x1": data[:, 0].round(2),
        "y1": data[:, 1].round(2),
        "x2": data[:, 2].round(2),
        "y2": data[:, 3].round(2),
    })

In [ ]:
import shutil
from pathlib import Path

MODELS_DIR = ASSETS / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

def load_pretrained_yolo(model_name="yolo26n.pt"):
    """学習済みモデルを読み込む。初回だけダウンロードする。
    Load a pretrained model, downloading it only on the first run."""
    from ultralytics import YOLO

    weights_path = MODELS_DIR / model_name
    if not weights_path.is_file():
        print(f"初回のみダウンロードします / downloading {model_name} (first run only)...")
        print("  ネットワークに接続できない場合は、次を確認してください:")
        print("  - 社内プロキシの設定 / corporate proxy settings")
        print("  - もう一度このセルを実行する / run this cell again")
        model = YOLO(model_name)          # ダウンロードはカレントフォルダに行われる
        downloaded = Path(model_name)
        if downloaded.is_file():
            shutil.move(str(downloaded), str(weights_path))
            print("保存しました / cached to:", weights_path)
        else:
            raise RuntimeError(
                f"モデルを取得できませんでした / could not obtain {model_name}. "
                "ネットワークを確認して再実行してください / check the network and retry."
            )
    return YOLO(str(weights_path))

---

## Part 1　学習とは何か / What training is

#### 日本語：学習の繰り返し
**日本語：** 学習は、次の 4 ステップの **繰り返し** です。

1. **予測する** — 今のモデルで、学習画像を推論する
2. **間違いを測る** — 正解との差を数値にする。これを **損失（loss）** と呼ぶ
3. **少し直す** — 損失が小さくなる方向に、モデルの数値を少し動かす
4. **繰り返す** — 1 周分（全学習画像）を終えると 1 **エポック**

**日本語：** 損失（loss）が下がっていれば、学習は進んでいます。
ただし **学習用の損失が下がり続けても、実力が上がるとは限りません**（過学習）。

**English:** Training is a **loop** of four steps:

1. **predict** — run the current model on the training images
2. **measure the error** — compare with the truth; the number is the **loss**
3. **adjust a little** — nudge the model's numbers in the direction that reduces the loss
4. **repeat** — one full pass over the training set is one **epoch**

If the loss falls, learning is progressing. But a falling *training* loss does not guarantee better
real-world skill — that is overfitting.
#### English: The training loop

#### 日本語：用語を一度に整理する
**日本語：**

| 用語 | 意味 | この研修での値 |
|---|---|---|
| エポック（epoch） | 全学習画像を 1 周すること | 15 |
| バッチサイズ（batch） | 一度に処理する画像枚数 | 8 |
| 学習率（learning rate） | 1 回の修正の大きさ | Ultralytics の既定値 |
| 損失（loss） | 正解との差。小さいほど良い | 学習で下がる |
| 学習（train） | 数値を決める作業 | — |
| 検証（val） | 学習中に実力を確認する | 各エポック後 |

**日本語：** **バッチサイズを大きくすると 1 回の処理が重くなり、小さくすると軽くなります**
（ただし不安定になりやすい）。メモリが足りないときは小さくします。

**English:**

| Term | Meaning | Value in this course |
|---|---|---|
| epoch | one full pass over the training set | 15 |
| batch size | images processed at once | 8 |
| learning rate | size of each adjustment | Ultralytics default |
| loss | distance from the truth; lower is better | falls during training |
| train | the work that determines the numbers | — |
| val | checking real skill during training | after each epoch |

A larger batch is heavier but more stable; a smaller batch is lighter but noisier. Reduce the batch
when you run out of memory.
#### English: The terms, all at once

#### 日本語：ファインチューニング
**日本語：** ゼロから学習するのではなく、**COCO で学習済みの `yolo26n.pt` から始めます**。
これを **ファインチューニング** と呼びます。

理由：COCO で学んだ「物の形を見る力」は、PVC の欠陥にも役立ちます。
少ない画像（64 枚）でも、形の見方を一から覚え直す必要がありません。

**日本語：** とはいえ、**64 枚は非常に少ない** です。本番では数百〜数千枚が必要になります。

**English:** Instead of training from scratch we **start from `yolo26n.pt`, pretrained on COCO**.
This is **fine-tuning**.

Reason: the "shape understanding" learned on COCO transfers usefully to PVC defects, so even with
64 images the model does not have to relearn how to see.

Still, **64 images is very few**. Real deployment needs hundreds to thousands.
#### English: Fine-tuning

---

## Part 2　学習の準備 / Preparing the run

#### 日本語：`data.yaml` を今の場所に合わせる
**日本語：** `data.yaml` の `path` は、データセット生成時の絶対パスです。
フォルダを移動した場合は、**今の場所に書き直す** 必要があります。

下のセルは、`data.yaml` を **今の研修フォルダに合わせて書き直します**。
他の設定（クラス名など）はそのまま残します。

**English:** `path` in `data.yaml` is the absolute path from generation time. If you moved the
folder, it must be rewritten.

The next cell **rewrites `data.yaml` for the current course folder** while keeping every other
setting.
#### English: Point `data.yaml` at the current location

In [ ]:
dataset_dir = ASSETS / "yolo_dataset"
data_yaml_path = dataset_dir / "data.yaml"

# 既存の設定を読み、path の行だけを今の場所に置き換える
# keep every line, but replace the "path:" line with the current location
lines = data_yaml_path.read_text(encoding="utf-8").splitlines()
updated = []
for line in lines:
    if line.startswith("path:"):
        updated.append(f"path: {dataset_dir.as_posix()}")
    else:
        updated.append(line)
data_yaml_path.write_text("\n".join(updated) + "\n", encoding="utf-8")

print(data_yaml_path.read_text(encoding="utf-8"))
print("data.yaml を更新しました / updated:", data_yaml_path)

#### 日本語：学習画像とラベルの対応を最終確認する
**日本語：** 学習を始める前に、**画像とラベルが 1 対 1 で対応しているか** を確認します。
ここで失敗すると、学習が始まってから変な結果になります。

**English:** Before training, confirm that **every image has its label**. Failing here produces
strange results only after training has run.
#### English: Final check that images and labels pair up

In [ ]:
report = []
for split in ("train", "val", "test"):
    images = {p.stem for p in (dataset_dir / "images" / split).glob("*.png")}
    labels = {p.stem for p in (dataset_dir / "labels" / split).glob("*.txt")}
    report.append({
        "split": split,
        "images": len(images),
        "labels": len(labels),
        "missing_labels": len(images - labels),
        "orphan_labels": len(labels - images),
    })

import pandas as pd
print(pd.DataFrame(report).to_string(index=False))

---

## Part 3　学習を実行する / Running the training

#### 日本語：このセルの学習設定
**日本語：** CPU でも現実的な時間で終わる、**小さな設定** にしています。

| 設定 | 値 | 理由 |
|---|---|---|
| `epochs` | 15 | 少なすぎず、待ち時間も短い |
| `imgsz` | 320 | 元画像が 320 px。CPU でも扱える |
| `batch` | 8 | メモリと速度のバランス |
| `workers` | 0 | Windows のノートブックで安全 |
| `device` | `cpu` | GPU が無くても動く |
| `seed` | 2026 | 再現性のため |

> **日本語：** **待ち時間はハードウェアで大きく変わります。** 数分で終わることもあれば、
> 15 分以上かかることもあります。**終了時間は保証できません。**
> 遅い場合は `epochs` を 5 に下げてください（学習の流れは同じです）。

**English:** The configuration is deliberately **small** so it finishes in a reasonable time on CPU.

| Setting | Value | Why |
|---|---|---|
| `epochs` | 15 | enough to see progress, short enough to wait |
| `imgsz` | 320 | the source images are 320 px; CPU friendly |
| `batch` | 8 | balance between memory and speed |
| `workers` | 0 | safe in Windows notebooks |
| `device` | `cpu` | works without a GPU |
| `seed` | 2026 | reproducibility |

**Runtime depends heavily on the hardware**: minutes on some machines, 15+ minutes on others.
**Completion time cannot be guaranteed.** If it is too slow, set `epochs = 5`; the learning process
is the same.
#### English: The configuration in this cell

#### 日本語：実行スイッチ
**日本語：** 次のセルを実行すると **学習が始まります**（CPU で数分かかります）。
スキップしたい場合は `RUN_TRAINING = False` に書き換えてください。

**長い学習（GPU 向け 100 エポック）は実行しません。** 例として文字列に保存してあります。

**English:** Running the next cell **starts a short training job** (a few minutes on CPU).
Set `RUN_TRAINING = False` to skip it.

**The long GPU configuration (100 epochs) is never executed**; it is stored only as a string example.
#### English: The run switch

In [ ]:
# ================= 実行スイッチ / run switch =================
RUN_TRAINING = True         # 学習をスキップするときは False にしてください / set to False to skip
EPOCHS = 15                 # 遅い場合は 5 に下げてください / lower to 5 if it is slow
IMGSZ = 320
BATCH = 8
SEED = 2026
# =============================================================

RUN_LONG_GPU_EXAMPLE = """
# GPU がある場合の例（このノートブックでは実行しません）
# example for a machine with a GPU (not executed here)
model = YOLO("yolo26n.pt")
model.train(
    data=data_yaml_path.as_posix(),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,          # 0 = first GPU
    workers=4,
    seed=SEED,
    project=str(COURSE_ROOT / "training_results" / "day04"),
    name="pvc_defect_long",
)
"""

if not RUN_TRAINING:
    print("学習は実行していません。/ Training was not started.")
    print("理由 / reason: RUN_TRAINING = False")
    print()
    print("実行するには、このセルの RUN_TRAINING を True に書き換えて再実行してください。")
    print("Set RUN_TRAINING = True in this cell and run it again.")
    print("GPU 向けの長い設定の例は RUN_LONG_GPU_EXAMPLE に入っています（実行はしません）。")
    print()
    print("スキップした場合、以降のセルは動きません。第4日ノートブック03 も実行できません。")
    print("If you skip training, the following cells cannot run, nor can Day 4 Notebook 03.")
else:
    print("学習を開始します / starting training...")
    print(f"epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} seed={SEED}")

In [ ]:
# 学習の実行 / run the training
if RUN_TRAINING:
    import time

    train_model = load_pretrained_yolo("yolo26n.pt")
    started = time.time()

    train_results = train_model.train(
        data=data_yaml_path.as_posix(),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=0,                 # Windows のノートブックでは 0 が安全 / safe on Windows
        device="cpu",
        seed=SEED,
        project=str(COURSE_ROOT / "training_results" / "day04"),
        name="pvc_defect_quick",
        exist_ok=True,
        plots=True,
        verbose=False,
    )

    elapsed_minutes = (time.time() - started) / 60
    run_dir = Path(train_model.trainer.save_dir)
    print()
    print(f"学習にかかった時間 / training time: {elapsed_minutes:.1f} minutes")
    print("出力フォルダ / output folder:", run_dir)
else:
    run_dir = None
    print("RUN_TRAINING = False のためスキップしました / skipped because RUN_TRAINING is False")

#### 日本語：学習中の画面の読み方
**日本語：** `verbose=True` にすると、学習中にエポックごとの行が出ます（この研修では静かにするため `False` にしています）。

```text
Epoch   GPU_mem  box_loss  cls_loss  dfl_loss  Instances  Size
      1/15    0.0G    1.842     2.913     1.394         18   320
```

| 列 | 意味 |
|---|---|
| `Epoch` | 何周目か |
| `box_loss` | 四角の位置の誤差。下がれば位置が合ってきた |
| `cls_loss` | クラス分類の誤差。下がれば種類を当てられるようになった |
| `dfl_loss` | 境界の細かさの誤差 |
| `Instances` | そのバッチに含まれた欠陥の数 |

**日本語：** 損失が下がり続けていれば学習は進んでいます。
**下がらない／上がる場合は、学習率やデータを疑います。**

**English:** With `verbose=True` you see one line per epoch (this course sets `False` to keep the output quiet):

```text
Epoch   GPU_mem  box_loss  cls_loss  dfl_loss  Instances  Size
      1/15    0.0G    1.842     2.913     1.394         18   320
```

| Column | Meaning |
|---|---|
| `Epoch` | which pass |
| `box_loss` | box position error; falling means boxes fit better |
| `cls_loss` | class error; falling means classes are better |
| `dfl_loss` | boundary refinement error |
| `Instances` | number of defects in that batch |

Falling losses mean learning is working. Flat or rising losses point to the learning rate or the data.
#### English: How to read the training output

---

## Part 4　学習結果を保存する / Saving the results

#### 日本語：4 つを保存する
**日本語：** 学習が終わると、Ultralytics が出力フォルダに多くのファイルを作ります。
提出物としては、次の 4 つがあれば十分です。

| 種類 | ファイル | 用途 |
|---|---|---|
| 重み | `weights/best.pt` | 推論に使う。**最重要** |
| 設定 | `args.yaml` | どんな設定で学習したかの記録 |
| 指標 | `results.csv` | エポックごとの損失と精度 |
| ログ | `results.csv` から作る要約 | 提出用 |

**日本語：** `best.pt` は検証データで最も良かった時点の重み、`last.pt` は最後の重みです。
**ふつうは `best.pt` を使います。**

**English:** Ultralytics writes many files into the output folder. For submission you need four:

| Kind | File | Use |
|---|---|---|
| weights | `weights/best.pt` | for inference; **the important one** |
| config | `args.yaml` | the record of how training was configured |
| metrics | `results.csv` | loss and accuracy per epoch |
| log | a summary built from `results.csv` | for submission |

`best.pt` is the best checkpoint measured on the validation split; `last.pt` is the final one.
**Normally you use `best.pt`.**
#### English: Save four things

In [ ]:
if run_dir is not None:
    best_weights = run_dir / "weights" / "best.pt"
    last_weights = run_dir / "weights" / "last.pt"
    args_file = run_dir / "args.yaml"
    results_csv = run_dir / "results.csv"

    for label, path in [("best.pt", best_weights), ("last.pt", last_weights),
                        ("args.yaml", args_file), ("results.csv", results_csv)]:
        exists = path.is_file()
        size = path.stat().st_size if exists else 0
        print(f"{label:<12} exists={exists}  size={size} bytes")

    # 学習結果を提出用フォルダにコピーする / copy the results for submission
    submission_dir = COURSE_ROOT / "submissions" / "day04"
    submission_dir.mkdir(parents=True, exist_ok=True)
    if best_weights.is_file():
        import shutil
        shutil.copyfile(best_weights, submission_dir / "best.pt")
        print()
        print("提出用にコピーしました / copied for submission:", submission_dir / "best.pt")
else:
    print("学習を実行していないため、保存するものはありません。")
    print("Nothing to save because training was not run.")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
学習を開始します / starting training...
epochs=15 imgsz=320 batch=8 seed=2026

学習にかかった時間 / training time: 2.3 minutes
出力フォルダ / output folder: .../training_results/day04/pvc_defect_quick

best.pt      exists=True  size=5416133 bytes
last.pt      exists=True  size=5416133 bytes
args.yaml    exists=True  size=2091 bytes
results.csv  exists=True  size=2062 bytes

（サイズは環境により変わります / file sizes vary between machines）
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
starting training...
epochs=15 imgsz=320 batch=8 seed=2026

training time: 2.3 minutes
output folder: .../training_results/day04/pvc_defect_quick

best.pt      exists=True
last.pt      exists=True
args.yaml    exists=True
results.csv  exists=True
```

In [ ]:
if run_dir is not None and results_csv.is_file():
    history = pd.read_csv(results_csv)
    history.columns = [c.strip() for c in history.columns]
    print("列 / columns:", list(history.columns))
    print()
    print(history.tail(5).to_string(index=False))

    # 提出用の要約 / a short summary for the submission
    summary_lines = [
        "# 学習ログの要約 / training log summary",
        "",
        f"epochs={EPOCHS} imgsz={IMGSZ} batch={BATCH} seed={SEED}",
        "",
        history.to_string(index=False),
    ]
    summary_path = COURSE_ROOT / "submissions" / "day04" / "training_log_summary.txt"
    summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
    print()
    print("保存 / saved:", summary_path)
else:
    print("results.csv がありません / results.csv is not available")

#### 日本語：学習は成功したと言えるか
**日本語：** 損失が下がっただけでは不十分です。**まだ何も確認していません。**

次のノートブック（04_03）で、
**学習に使っていない画像** で性能を測ります。そこで初めて「使える／使えない」が分かります。

**日本語：** この段階で「精度 95 %」のような数字を出すことは **しません**。
学習用データで測った数字は、実力を表さないからです。

**English:** Falling losses are not enough — **nothing has been verified yet**.

The next notebook (04_03) measures performance on **images never used for training**. Only then can
you say whether the model is usable.

We deliberately do **not** quote an accuracy figure here: a number measured on training data does
not reflect real skill.
#### English: Can we call this training successful?

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：学習を 5 エポックで実行する
**日本語：** `EPOCHS` を 5 にして学習を実行し、`box_loss` の最初と最後の値を記録してください。

| エポック | box_loss |
|---|---|
| 1 | |
| 5 | |

**English:** Set `EPOCHS = 5`, run the training, and record the first and last `box_loss` values.
#### English: Exercise 1 (basic): train for 5 epochs
Done when both values are recorded and you can say whether the loss decreased.

In [ ]:
# 演習 1 / Exercise 1
# EPOCHS = 5 にして、上の学習セルを再実行してください / set EPOCHS = 5 and re-run the training cell
# 記録 / record:
first_box_loss = None
last_box_loss = None
print("epoch 1 box_loss:", first_box_loss)
print("epoch 5 box_loss:", last_box_loss)

#### 日本語：演習 2（中級）：`imgsz` を変えて比べる
**日本語：** `IMGSZ` を 160 / 320 で学習し、**学習時間** と **`box_loss` の最終値** を比べてください。
速さと精度のどちらを取るか、自分の言葉で書いてください。

**English:** Train with `IMGSZ` 160 and 320, compare **training time** and the **final `box_loss`**,
and state which trade-off you would choose and why.
#### English: Exercise 2 (intermediate): compare `imgsz`
Done when both runs are recorded and the trade-off is explained.

In [ ]:
# 演習 2 / Exercise 2
comparison = [
    {"imgsz": 160, "minutes": None, "final_box_loss": None},
    {"imgsz": 320, "minutes": None, "final_box_loss": None},
]
for row in comparison:
    print(row)

#### 日本語：演習 3（発展）：学習が進まない原因を考える
**日本語：** 損失が下がらない、または NaN になる場合の原因を 4 つ挙げ、
それぞれ「どう確認するか」を書いてください。

**English:** List four reasons why the loss might not decrease (or becomes NaN), and for each one
say how you would check it.
#### English: Exercise 3 (challenge): why training might not progress
Done when four causes and four checks are written.

In [ ]:
# 演習 3 / Exercise 3
causes = [
    ("", ""),
    ("", ""),
    ("", ""),
    ("", ""),
]
for index, (cause, check) in enumerate(causes, start=1):
    print(f"{index}. 原因 / cause: {cause or '-'} -> 確認 / check: {check or '-'}")

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. エポックとバッチサイズの違いを説明してください。
2. 損失（loss）が下がるとは、どういうことですか。
3. ファインチューニングをする利点を 2 つ挙げてください。
4. `best.pt` と `last.pt` の違いは何ですか。どちらを使いますか。
5. 学習が終わった直後に「精度は 95 % です」と言ってはいけないのはなぜですか。

**English:**

1. Explain the difference between an epoch and a batch size.
2. What does it mean that the loss decreases?
3. Give two benefits of fine-tuning.
4. What is the difference between `best.pt` and `last.pt`, and which do you use?
5. Why must you not announce "accuracy is 95%" right after training finishes?
#### English: Understanding questions
Model answers: `docs/instructor/day04_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| 学習が始まらない | Training does not start | `RUN_TRAINING = False` になっています。学習するには `True` にしてください。 | `RUN_TRAINING` is still False. Set it to True and re-run. |
| `RuntimeError: Dataset ... not found` | `RuntimeError: Dataset ... not found` | `data.yaml` の `path` が古い可能性があります。Part 2 のセルで書き直してください。 | `path` in `data.yaml` may be stale. Re-run the Part 2 cell. |
| `CUDA out of memory` / `OutOfMemoryError` | `CUDA out of memory` / `OutOfMemoryError` | `BATCH` を 4、`IMGSZ` を 224 に下げてください。CPU 実行なら起きにくいです。 | Lower `BATCH` to 4 and `IMGSZ` to 224. Rare when running on CPU. |
| 学習が非常に遅い | Training is very slow | CPU 実行では数分〜十数分かかります。`EPOCHS` を 5、`IMGSZ` を 160 にしてください。 | CPU training takes minutes to tens of minutes. Use `EPOCHS=5`, `IMGSZ=160`. |
| `FileNotFoundError: yolo26n.pt` | `FileNotFoundError: yolo26n.pt` | モデルが未取得です。`assets/models/` にファイルがあるか確認してください。 | The model was never obtained. Check `assets/models/`. |
| 損失が下がらない | The loss does not decrease | ラベルの誤り、画像とラベルの不一致、学習率の問題などが考えられます。第4日01 の点検に戻ってください。 | Likely label errors, image/label mismatch, or the learning rate. Repeat the Day 4 Notebook 01 checks. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 4 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-4 and ran every code cell
- [ ] `data.yaml` を今の場所に合わせて書き直した
      - [ ] Rewrote `data.yaml` for the current location
- [ ] 画像とラベルの対応を確認した
      - [ ] Confirmed images and labels pair up
- [ ] 学習を実行した（`RUN_TRAINING = True`）
      - [ ] Ran the training (`RUN_TRAINING = True`)
- [ ] `best.pt` と `results.csv` が生成された
      - [ ] `best.pt` and `results.csv` exist
- [ ] 学習ログの要約を保存した
      - [ ] Saved the training summary
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Ultralytics Train モード](https://docs.ultralytics.com/modes/train/)
- [Ultralytics ハイパーパラメータ](https://docs.ultralytics.com/guides/hyperparameter-tuning/)
- [PyTorch 学習の基礎](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** `BATCH` を 4 と 16 で学習し、損失の下がり方の「なめらかさ」を比べてください。

**English:** Train with `BATCH` 4 and 16 and compare how smoothly the loss decreases.